In [107]:
import pandas as pd
import numpy as np
import ast
from google.colab import files
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import joblib

In [108]:
movies=pd.read_csv('tmdb_5000_movies.csv')
credits=pd.read_csv('tmdb_5000_credits.csv')

In [109]:
print("Movies shape:", movies.shape)
print("Credits shape:", credits.shape)

Movies shape: (4803, 20)
Credits shape: (4803, 4)


In [110]:
movies.head()

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500
2,245000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.sonypictures.com/movies/spectre/,206647,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""nam...","[{""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""...",2015-10-26,880674609,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,A Plan No One Escapes,Spectre,6.3,4466
3,250000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...",http://www.thedarkknightrises.com/,49026,"[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...",en,The Dark Knight Rises,Following the death of District Attorney Harve...,112.312950,"[{""name"": ""Legendary Pictures"", ""id"": 923}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-07-16,1084939099,165.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,The Legend Ends,The Dark Knight Rises,7.6,9106
4,260000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://movies.disney.com/john-carter,49529,"[{""id"": 818, ""name"": ""based on novel""}, {""id"":...",en,John Carter,"John Carter is a war-weary, former military ca...",43.926995,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}]","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-03-07,284139100,132.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"Lost in our world, found in another.",John Carter,6.1,2124


In [111]:
credits.head()

,movie_id,title,cast,crew
0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,206647,Spectre,"[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."
3,49026,The Dark Knight Rises,"[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de..."
4,49529,John Carter,"[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de..."


In [112]:
print("Movies columns:")
print(movies.columns.tolist())

print("\nCredits columns:")
print(credits.columns.tolist())

Movies columns:
['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language', 'original_title', 'overview', 'popularity', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'spoken_languages', 'status', 'tagline', 'title', 'vote_average', 'vote_count']

Credits columns:
['movie_id', 'title', 'cast', 'crew']


In [113]:
movies = movies.merge(credits, on="title")

In [114]:
print("Merged dataset shape :",movies.shape)
print("\nColumns:")
print(movies.columns.tolist())

Merged dataset shape : (4809, 23)

Columns:
['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language', 'original_title', 'overview', 'popularity', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'spoken_languages', 'status', 'tagline', 'title', 'vote_average', 'vote_count', 'movie_id', 'cast', 'crew']


In [115]:
movies.filter(like="movie_id").head()

,movie_id
0,19995
1,285
2,206647
3,49026
4,49529


In [116]:
movies=movies[['movie_id','title','overview','genres','keywords','cast','crew']]

In [117]:
movies.head()

,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...","[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...","[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,206647,Spectre,A cryptic message from Bond’s past sends him o...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...","[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."
3,49026,The Dark Knight Rises,Following the death of District Attorney Harve...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...","[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...","[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de..."
4,49529,John Carter,"John Carter is a war-weary, former military ca...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 818, ""name"": ""based on novel""}, {""id"":...","[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de..."


In [118]:
print("Dataset shape:", movies.shape)
print("Selected columns:", movies.columns.tolist())

Dataset shape: (4809, 7)
Selected columns: ['movie_id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew']


In [119]:
print("Missing Values:")
print(movies.isnull().sum())
print("\nDuplicate Values:")
print(movies.duplicated().sum())
print("\nDuplicate titles:", movies["title"].duplicated().sum())

Missing Values:
movie_id    0
title       0
overview    3
genres      0
keywords    0
cast        0
crew        0
dtype: int64

Duplicate Values:
0

Duplicate titles: 9


In [120]:
movies.dropna(subset=['overview'],inplace=True)

In [121]:
movies.drop_duplicates(subset='title',inplace=True)
movies.drop_duplicates(subset='movie_id',inplace=True)
movies.reset_index(drop=True,inplace=True)

In [122]:
print("Shape after cleaning:", movies.shape)

print("\nMissing Values:")
print(movies.isnull().sum())

print("\nDuplicate movie IDs:", movies["movie_id"].duplicated().sum())
print("Duplicate titles:", movies["title"].duplicated().sum())

Shape after cleaning: (4797, 7)

Missing Values:
movie_id    0
title       0
overview    0
genres      0
keywords    0
cast        0
crew        0
dtype: int64

Duplicate movie IDs: 0
Duplicate titles: 0


In [123]:
print(movies.iloc[0]['genres'])
print(movies.iloc[0]['keywords'])

[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}, {"id": 14, "name": "Fantasy"}, {"id": 878, "name": "Science Fiction"}]
[{"id": 1463, "name": "culture clash"}, {"id": 2964, "name": "future"}, {"id": 3386, "name": "space war"}, {"id": 3388, "name": "space colony"}, {"id": 3679, "name": "society"}, {"id": 3801, "name": "space travel"}, {"id": 9685, "name": "futuristic"}, {"id": 9840, "name": "romance"}, {"id": 9882, "name": "space"}, {"id": 9951, "name": "alien"}, {"id": 10148, "name": "tribe"}, {"id": 10158, "name": "alien planet"}, {"id": 10987, "name": "cgi"}, {"id": 11399, "name": "marine"}, {"id": 13065, "name": "soldier"}, {"id": 14643, "name": "battle"}, {"id": 14720, "name": "love affair"}, {"id": 165431, "name": "anti war"}, {"id": 193554, "name": "power relations"}, {"id": 206690, "name": "mind and soul"}, {"id": 209714, "name": "3d"}]


In [124]:
def convert(text):
  names=[]
  for item in ast.literal_eval(text):
    names.append(item['name'])
  return names

In [125]:
movies['genres']=movies['genres'].apply(convert)
movies['keywords']=movies['keywords'].apply(convert)

In [126]:
movies[["title", "genres", "keywords"]].head()

,title,genres,keywords
0,Avatar,"[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon..."
1,Pirates of the Caribbean: At World's End,"[Adventure, Fantasy, Action]","[ocean, drug abuse, exotic island, east india ..."
2,Spectre,"[Action, Adventure, Crime]","[spy, based on novel, secret agent, sequel, mi..."
3,The Dark Knight Rises,"[Action, Crime, Drama, Thriller]","[dc comics, crime fighter, terrorist, secret i..."
4,John Carter,"[Action, Adventure, Science Fiction]","[based on novel, mars, medallion, space travel..."


In [127]:
movies.iloc[0]["cast"]

'[{"cast_id": 242, "character": "Jake Sully", "credit_id": "5602a8a7c3a3685532001c9a", "gender": 2, "id": 65731, "name": "Sam Worthington", "order": 0}, {"cast_id": 3, "character": "Neytiri", "credit_id": "52fe48009251416c750ac9cb", "gender": 1, "id": 8691, "name": "Zoe Saldana", "order": 1}, {"cast_id": 25, "character": "Dr. Grace Augustine", "credit_id": "52fe48009251416c750aca39", "gender": 1, "id": 10205, "name": "Sigourney Weaver", "order": 2}, {"cast_id": 4, "character": "Col. Quaritch", "credit_id": "52fe48009251416c750ac9cf", "gender": 2, "id": 32747, "name": "Stephen Lang", "order": 3}, {"cast_id": 5, "character": "Trudy Chacon", "credit_id": "52fe48009251416c750ac9d3", "gender": 1, "id": 17647, "name": "Michelle Rodriguez", "order": 4}, {"cast_id": 8, "character": "Selfridge", "credit_id": "52fe48009251416c750ac9e1", "gender": 2, "id": 1771, "name": "Giovanni Ribisi", "order": 5}, {"cast_id": 7, "character": "Norm Spellman", "credit_id": "52fe48009251416c750ac9dd", "gender": 

In [128]:
def convert_cast(text):
  names=[]
  for val in ast.literal_eval(text):
    if(len(names)<3):
      names.append(val['name'])
    else:
      break
  return names

In [129]:
movies['cast']=movies['cast'].apply(convert_cast)


In [130]:
def fetch_director(text):
  directors=[]
  for val in ast.literal_eval(text):
    if val['job']=="Director":
      directors.append(val['name'])
      break
  return directors

In [131]:
movies['crew']=movies['crew'].apply(fetch_director)

In [132]:
movies[['title','crew','cast']].head()

,title,crew,cast
0,Avatar,[James Cameron],"[Sam Worthington, Zoe Saldana, Sigourney Weaver]"
1,Pirates of the Caribbean: At World's End,[Gore Verbinski],"[Johnny Depp, Orlando Bloom, Keira Knightley]"
2,Spectre,[Sam Mendes],"[Daniel Craig, Christoph Waltz, Léa Seydoux]"
3,The Dark Knight Rises,[Christopher Nolan],"[Christian Bale, Michael Caine, Gary Oldman]"
4,John Carter,[Andrew Stanton],"[Taylor Kitsch, Lynn Collins, Samantha Morton]"


In [133]:
movies['overview']=movies['overview'].apply(lambda x:x.split())

In [134]:
def remove_spaces(text):
  return [i.replace(" ","") for i in text]

In [135]:
movies['genres']=movies['genres'].apply(remove_spaces)
movies['keywords']=movies['keywords'].apply(remove_spaces)
movies['cast']=movies['cast'].apply(remove_spaces)
movies['crew']=movies['crew'].apply(remove_spaces)

In [136]:
movies[['overview','genres','keywords','cast','crew']].head()

,overview,genres,keywords,cast,crew
0,"[In, the, 22nd, century,, a, paraplegic, Marin...","[Action, Adventure, Fantasy, ScienceFiction]","[cultureclash, future, spacewar, spacecolony, ...","[SamWorthington, ZoeSaldana, SigourneyWeaver]",[JamesCameron]
1,"[Captain, Barbossa,, long, believed, to, be, d...","[Adventure, Fantasy, Action]","[ocean, drugabuse, exoticisland, eastindiatrad...","[JohnnyDepp, OrlandoBloom, KeiraKnightley]",[GoreVerbinski]
2,"[A, cryptic, message, from, Bond’s, past, send...","[Action, Adventure, Crime]","[spy, basedonnovel, secretagent, sequel, mi6, ...","[DanielCraig, ChristophWaltz, LéaSeydoux]",[SamMendes]
3,"[Following, the, death, of, District, Attorney...","[Action, Crime, Drama, Thriller]","[dccomics, crimefighter, terrorist, secretiden...","[ChristianBale, MichaelCaine, GaryOldman]",[ChristopherNolan]
4,"[John, Carter, is, a, war-weary,, former, mili...","[Action, Adventure, ScienceFiction]","[basedonnovel, mars, medallion, spacetravel, p...","[TaylorKitsch, LynnCollins, SamanthaMorton]",[AndrewStanton]


In [137]:
movies["tags"] = (
    movies["overview"]
    + movies["genres"]
    + movies["keywords"]
    + movies["cast"]
    + movies["crew"]
)

In [138]:
movies[["title", "tags"]].head()

,title,tags
0,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin..."
1,Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d..."
2,Spectre,"[A, cryptic, message, from, Bond’s, past, send..."
3,The Dark Knight Rises,"[Following, the, death, of, District, Attorney..."
4,John Carter,"[John, Carter, is, a, war-weary,, former, mili..."


In [139]:
movies['tags']=movies['tags'].apply(lambda x: " ".join(x).lower())

In [140]:
print(movies.iloc[0]["tags"])

in the 22nd century, a paraplegic marine is dispatched to the moon pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization. action adventure fantasy sciencefiction cultureclash future spacewar spacecolony society spacetravel futuristic romance space alien tribe alienplanet cgi marine soldier battle loveaffair antiwar powerrelations mindandsoul 3d samworthington zoesaldana sigourneyweaver jamescameron


In [141]:
new_movies=movies[['movie_id','title','tags']]

In [142]:
new_movies.head()

,movie_id,title,tags
0,19995,Avatar,"in the 22nd century, a paraplegic marine is di..."
1,285,Pirates of the Caribbean: At World's End,"captain barbossa, long believed to be dead, ha..."
2,206647,Spectre,a cryptic message from bond’s past sends him o...
3,49026,The Dark Knight Rises,following the death of district attorney harve...
4,49529,John Carter,"john carter is a war-weary, former military ca..."


In [143]:
from nltk.stem.porter import PorterStemmer
stemmer = PorterStemmer()

In [144]:
def stem_text(text):
  stemmed_words=[]
  for words in text.split():
    stemmed_words.append(stemmer.stem(words))
  return " ".join(stemmed_words)

In [145]:
new_movies['tags']=new_movies['tags'].apply(stem_text)

/tmp/ipykernel_1183/2852466781.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_movies['tags']=new_movies['tags'].apply(stem_text)


In [146]:
print(new_movies.iloc[0]["tags"])

in the 22nd century, a parapleg marin is dispatch to the moon pandora on a uniqu mission, but becom torn between follow order and protect an alien civilization. action adventur fantasi sciencefict cultureclash futur spacewar spacecoloni societi spacetravel futurist romanc space alien tribe alienplanet cgi marin soldier battl loveaffair antiwar powerrel mindandsoul 3d samworthington zoesaldana sigourneyweav jamescameron


In [147]:
vectorizer=CountVectorizer(
        max_features=5000,
        stop_words="english"
)

In [148]:
vectors=vectorizer.fit_transform(new_movies['tags']).toarray()

In [149]:
print("Vectors shape:", vectors.shape)

Vectors shape: (4797, 5000)


In [150]:
print(vectorizer.get_feature_names_out())

['000' '007' '10' ... 'zone' 'zoo' 'zooeydeschanel']


In [151]:
print(vectors[0])

[0 0 0 ... 0 0 0]


In [152]:
similarity = cosine_similarity(vectors)

In [153]:
print(similarity[0])

[1.         0.08346223 0.0860309  ... 0.04499213 0.         0.        ]


In [154]:
sorted(
    list(enumerate(similarity[0])),
    key=lambda item: item[1],
    reverse=True
)[:6]

[(0, np.float64(1.0000000000000002)),
 (1213, np.float64(0.28676966733820225)),
 (2403, np.float64(0.26901379342448517)),
 (3721, np.float64(0.2605130246476754)),
 (507, np.float64(0.255608593705383)),
 (539, np.float64(0.25038669783359574))]

In [155]:
def recommend(movie_title):
    movie_index = new_movies[
        new_movies["title"] == movie_title
    ].index[0]

    movie_scores = similarity[movie_index]

    similar_movies = sorted(
        list(enumerate(movie_scores)),
        key=lambda item: item[1],
        reverse=True
    )[1:6]

    for index, score in similar_movies:
        print(
            new_movies.iloc[index]["title"],
            "— Similarity:",
            round(score, 3)
        )

In [156]:
recommend("Avatar")

Aliens vs Predator: Requiem — Similarity: 0.287
Aliens — Similarity: 0.269
Falcon Rising — Similarity: 0.261
Independence Day — Similarity: 0.256
Titan A.E. — Similarity: 0.25


In [157]:
recommend("The Dark Knight")

The Dark Knight Rises — Similarity: 0.423
Batman Begins — Similarity: 0.402
Batman Returns — Similarity: 0.331
Batman Forever — Similarity: 0.294
Batman — Similarity: 0.261


In [158]:
tfidf=TfidfVectorizer(
    max_features=5000,
    stop_words="english"
)

In [159]:
tfidf_vectors=tfidf.fit_transform(new_movies['tags']).toarray()

In [160]:
tfidf_similarity=cosine_similarity(tfidf_vectors)

In [161]:
sorted(list(enumerate(tfidf_similarity[0])),reverse=True,key=lambda x:x[1])[1:6]

[(2403, np.float64(0.252152206628459)),
 (3721, np.float64(0.22565630877630477)),
 (582, np.float64(0.19407013148888339)),
 (1213, np.float64(0.18882873145259782)),
 (3602, np.float64(0.17700679105138595))]

In [162]:
def recommend_tfidf(movie_title):
    movie_index = new_movies[
        new_movies["title"] == movie_title
    ].index[0]

    movie_scores = tfidf_similarity[movie_index]

    similar_movies = sorted(
        list(enumerate(movie_scores)),
        key=lambda item: item[1],
        reverse=True
    )[1:6]

    for index, score in similar_movies:
        print(
            new_movies.iloc[index]["title"],
            "— Similarity:",
            round(score, 3)
        )

In [163]:
recommend_tfidf("Avatar")

Aliens — Similarity: 0.252
Falcon Rising — Similarity: 0.226
Battle: Los Angeles — Similarity: 0.194
Aliens vs Predator: Requiem — Similarity: 0.189
Apollo 18 — Similarity: 0.177


In [164]:
recommend_tfidf("The Dark Knight")

The Dark Knight Rises — Similarity: 0.461
Batman Returns — Similarity: 0.397
Batman Begins — Similarity: 0.365
Batman Forever — Similarity: 0.301
Batman: The Dark Knight Returns, Part 2 — Similarity: 0.287


In [165]:
recommend_tfidf("Superman")

Superman II — Similarity: 0.356
Superman Returns — Similarity: 0.297
Superman IV: The Quest for Peace — Similarity: 0.273
Superman III — Similarity: 0.261
Man of Steel — Similarity: 0.189


In [166]:
recommend("Superman")

Superman Returns — Similarity: 0.384
Superman II — Similarity: 0.379
Iron Man 2 — Similarity: 0.329
Superman III — Similarity: 0.323
Superman IV: The Quest for Peace — Similarity: 0.311


In [167]:
def get_top_recommendations(movie_title, similarity_matrix, top_n=5):
    matching_movie = new_movies[
        new_movies["title"] == movie_title
    ]

    if matching_movie.empty:
        return []

    movie_index = matching_movie.index[0]

    similar_movies = sorted(
        enumerate(similarity_matrix[movie_index]),
        key=lambda item: item[1],
        reverse=True
    )[1:top_n + 1]

    results = []

    for index, score in similar_movies:
        results.append(
            (
                new_movies.iloc[index]["title"],
                round(float(score), 3)
            )
        )

    return results

In [168]:
def compare_models(movie_title):
    count_results = get_top_recommendations(
        movie_title,
        similarity
    )

    tfidf_results = get_top_recommendations(
        movie_title,
        tfidf_similarity
    )

    comparison = pd.DataFrame({
        "CountVectorizer Movie": [
            movie for movie, score in count_results
        ],
        "Count Score": [
            score for movie, score in count_results
        ],
        "TF-IDF Movie": [
            movie for movie, score in tfidf_results
        ],
        "TF-IDF Score": [
            score for movie, score in tfidf_results
        ]
    })

    return comparison

In [169]:
compare_models("Avatar")

,CountVectorizer Movie,Count Score,TF-IDF Movie,TF-IDF Score
0,Aliens vs Predator: Requiem,0.287,Aliens,0.252
1,Aliens,0.269,Falcon Rising,0.226
2,Falcon Rising,0.261,Battle: Los Angeles,0.194
3,Independence Day,0.256,Aliens vs Predator: Requiem,0.189
4,Titan A.E.,0.250,Apollo 18,0.177


In [170]:
compare_models("Iron Man")

,CountVectorizer Movie,Count Score,TF-IDF Movie,TF-IDF Score
0,Iron Man 3,0.437,Iron Man 2,0.465
1,Iron Man 2,0.404,Iron Man 3,0.399
2,Avengers: Age of Ultron,0.336,Avengers: Age of Ultron,0.301
3,The Avengers,0.271,Ant-Man,0.208
4,Captain America: Civil War,0.264,Captain America: Civil War,0.203


In [171]:
joblib.dump(
    new_movies,
    "movies.pkl"
)

joblib.dump(
    tfidf_vectors,
    "tfidf_vectors.pkl"
)

joblib.dump(
    tfidf,
    "tfidf_vectorizer.pkl"
)

['tfidf_vectorizer.pkl']

In [172]:
new_movies.to_csv("new_movies.csv",index=False)

In [174]:
%%writefile app.py
import joblib
import streamlit as st
from sklearn.metrics.pairwise import cosine_similarity


# --------------------------------------------------
# Page configuration
# --------------------------------------------------
st.set_page_config(
    page_title="Movie Recommendation System",
    page_icon="🎬",
    layout="wide"
)


# --------------------------------------------------
# Load saved artifacts
# --------------------------------------------------
@st.cache_resource
def load_artifacts():
    loaded_movies = joblib.load("movies.pkl")
    loaded_vectors = joblib.load("tfidf_vectors.pkl")

    # DataFrame positions ko vectors ke saath align rakhega
    loaded_movies = loaded_movies.reset_index(drop=True)

    return loaded_movies, loaded_vectors


try:
    movies, tfidf_vectors = load_artifacts()

except FileNotFoundError as error:
    st.error(
        "Required files nahi mili. Ensure karo ki "
        "movies.pkl aur tfidf_vectors.pkl, app.py ke "
        "same folder mein present hain."
    )
    st.exception(error)
    st.stop()

except Exception as error:
    st.error("Saved model artifacts load nahi ho paaye.")
    st.exception(error)
    st.stop()


# --------------------------------------------------
# Validate artifacts
# --------------------------------------------------
if len(movies) != tfidf_vectors.shape[0]:
    st.error(
        "Artifact mismatch: movies aur TF-IDF vectors "
        "ki rows equal nahi hain."
    )

    st.write("Total movies:", len(movies))
    st.write("Total vector rows:", tfidf_vectors.shape[0])

    st.stop()


required_columns = {"movie_id", "title"}

if not required_columns.issubset(movies.columns):
    st.error(
        "movies.pkl mein movie_id aur title columns "
        "available nahi hain."
    )
    st.stop()


# --------------------------------------------------
# Recommendation function
# --------------------------------------------------
def recommend(movie_title, top_n=5):
    matching_movies = movies[
        movies["title"] == movie_title
    ]

    if matching_movies.empty:
        return []

    movie_index = matching_movies.index[0]

    # Slicing ki wajah se shape (1, 5000) rahegi
    selected_movie_vector = tfidf_vectors[
        movie_index:movie_index + 1
    ]

    similarity_scores = cosine_similarity(
        selected_movie_vector,
        tfidf_vectors
    ).flatten()

    # Highest similarity score se lowest ki taraf
    sorted_indices = similarity_scores.argsort()[::-1]

    recommendations = []

    for index in sorted_indices:

        # Selected movie ko khud recommend nahi karna
        if index == movie_index:
            continue

        recommendations.append(
            {
                "movie_id": int(
                    movies.iloc[index]["movie_id"]
                ),
                "title": movies.iloc[index]["title"],
                "score": float(
                    similarity_scores[index]
                )
            }
        )

        if len(recommendations) == top_n:
            break

    return recommendations


# --------------------------------------------------
# Application heading
# --------------------------------------------------
st.title("🎬 Movie Recommendation System")

st.write(
    """
    Apni pasand ki movie select karo aur content-based
    filtering ki help se similar movies discover karo.
    """
)

st.caption(
    """
    Recommendations movie overview, genres, keywords,
    cast aur director ke basis par generate hoti hain.
    """
)

st.divider()


# --------------------------------------------------
# User inputs
# --------------------------------------------------
movie_titles = sorted(
    movies["title"]
    .dropna()
    .unique()
    .tolist()
)

selected_movie = st.selectbox(
    label="Choose a movie",
    options=movie_titles,
    index=None,
    placeholder="Search or select a movie"
)

number_of_recommendations = st.slider(
    label="Number of recommendations",
    min_value=3,
    max_value=10,
    value=5,
    step=1
)

recommend_button = st.button(
    label="Get Recommendations",
    type="primary",
    use_container_width=True
)


# --------------------------------------------------
# Generate and display recommendations
# --------------------------------------------------
if recommend_button:

    if selected_movie is None:
        st.warning("Please pehle koi movie select karo.")

    else:
        with st.spinner(
            "Similar movies find ki ja rahi hain..."
        ):
            recommendations = recommend(
                selected_movie,
                number_of_recommendations
            )

        if not recommendations:
            st.warning(
                "Selected movie ke liye recommendations "
                "nahi mili."
            )

        else:
            st.success(
                f"Recommendations based on: {selected_movie}"
            )

            st.subheader("Movies you may like")

            # Ek row mein 3 recommendation cards
            columns = st.columns(3)

            for position, movie in enumerate(
                recommendations
            ):
                column = columns[position % 3]

                with column:
                    with st.container(border=True):

                        st.caption(
                            f"RECOMMENDATION {position + 1}"
                        )

                        st.subheader(
                            movie["title"]
                        )

                        st.metric(
                            label="Similarity score",
                            value=f'{movie["score"]:.3f}'
                        )


            # Recommendation system explanation
            with st.expander(
                "How does this recommendation system work?"
            ):
                st.write(
                    """
                    First, every movie's overview, genres,
                    keywords, top cast members and director
                    are combined into a single tags column.
                    """
                )

                st.write(
                    """
                    TF-IDF converts these textual tags into
                    numerical vectors. Cosine similarity then
                    compares the selected movie's vector with
                    all other movie vectors.
                    """
                )

                st.info(
                    """
                    Similarity score content overlap represent
                    karta hai. Ye accuracy percentage, movie
                    rating ya success probability nahi hai.
                    """
                )


# --------------------------------------------------
# Model information
# --------------------------------------------------
st.divider()

with st.expander("Project details"):
    st.write("**Recommendation type:** Content-based filtering")
    st.write("**Text representation:** TF-IDF")
    st.write("**Similarity method:** Cosine similarity")
    st.write(f"**Movies available:** {len(movies):,}")
    st.write(f"**TF-IDF features:** {tfidf_vectors.shape[1]:,}")


# --------------------------------------------------
# Footer
# --------------------------------------------------
st.caption(
    "Built with Python, Pandas, Scikit-learn and Streamlit"
)

Overwriting app.py


In [175]:
import sklearn
import scipy
print(sklearn.__version__)
print(joblib.__version__)
print(pd.__version__)
print(np.__version__)
print(scipy.__version__)

1.6.1
1.5.3
2.2.2
2.0.2
1.16.3


In [176]:
%%writefile requirements.txt
scikit-learn==1.6.1
joblib==1.5.3
pandas==2.2.2
numpy==2.0.2
scipy==1.16.3
streamlit

Overwriting requirements.txt


In [177]:
!zip movie-recommendation-system.zip \
app.py \
movies.pkl \
new_movies.csv \
requirements.txt \
tfidf_vectorizer.pkl \
tfidf_vectors.pkl

updating: app.py (deflated 70%)
updating: movies.pkl (deflated 58%)
updating: new_movies.csv (deflated 57%)
updating: requirements.txt (deflated 12%)
updating: tfidf_vectorizer.pkl (deflated 71%)
updating: tfidf_vectors.pkl (deflated 99%)


In [ ]:
files.download("movie-recommendation-system.zip")